# 16 — Fog Enhancement Training

Train a compact supervised fog-image enhancement model for the Adaptive_Road_Vision pipeline.

This notebook uses real **Fog** images from the BDD100K training split as the source domain. Because the real fog subset is small, the training source is augmented with the **accepted synthetic fog images created by Notebooks 08–09 and included in the final training dataset**. Validation uses only real Fog images from the original validation split. Test Fog images remain untouched.

The clean target is generated as a deterministic pseudo-reference from each fog image. This is a reproducible training target, not human-annotated ground truth.

In [ ]:
from pathlib import Path
import os, json, random, math, time
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current, *list(current.parents[:5])]
    for candidate in candidates:
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate
    fallback = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
    if fallback.exists():
        return fallback
    return current

PROJECT_ROOT = find_project_root()

def resolve_dataset_root():
    for cand in [
        PROJECT_ROOT / "datasets" / "final_condition_balanced_v2",
        PROJECT_ROOT / "datasets" / "final_condition_balanced",
        PROJECT_ROOT / "datasets" / "final",
    ]:
        if (cand / "metadata" / "final_dataset_metadata.csv").exists():
            return cand
        if (cand / "final_dataset_metadata.csv").exists():
            return cand
        if (cand / "labels" / "train").exists() and any((cand / "labels" / "train").glob("*.txt")):
            return cand
    return PROJECT_ROOT / "datasets" / "final_condition_balanced_v2"

FINAL_DIR = resolve_dataset_root()
DATASET_ROOT = FINAL_DIR
RESULTS_DIR = PROJECT_ROOT / "results" / "enhancement"
MODEL_DIR = PROJECT_ROOT / "models" / "enhancement"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Results dir:", RESULTS_DIR)
print("Models dir:", MODEL_DIR)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("CUDA unavailable; training will use CPU.")

## 1. Configuration

The real Fog subset is small, so synthetic Fog training examples are enabled by default. Set `USE_SYNTHETIC_FOG = False` if you want a real-Fog-only experiment.

Synthetic Fog is training-only. The original validation and test Fog images are not used as synthetic sources or training targets.

In [ ]:
SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 0
EPOCHS = 30
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 7
SAVE_EVERY = 5

USE_SYNTHETIC_FOG = True
TRAIN_LIMIT = None
VAL_LIMIT = None

# When synthetic metadata is available, cap its contribution if desired.
SYNTHETIC_LIMIT = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "use_synthetic_fog": USE_SYNTHETIC_FOG,
    "device": str(DEVICE)
})

In [ ]:
metadata_candidates = [
    FINAL_DIR / "metadata" / "final_dataset_metadata.csv",
    FINAL_DIR / "final_dataset_metadata.csv",
    PROJECT_ROOT / "datasets" / "processed" / "metadata" / "bdd100k_processed_metadata.csv",
]
metadata_path = next((p for p in metadata_candidates if p.exists()), None)
if metadata_path is None:
    search_roots = [FINAL_DIR, PROJECT_ROOT / "datasets"]
    for root in search_roots:
        if root.exists():
            matches = list(root.glob("**/final_dataset_metadata.csv"))
            if matches:
                metadata_path = matches[0]
                break
assert metadata_path is not None, (
    "Could not find final_dataset_metadata.csv or processed metadata. "
    "Run Notebook 10 and verify the final dataset folder."
)

meta = pd.read_csv(metadata_path)
print("Metadata:", metadata_path)
print("Shape:", meta.shape)
print("Columns:", meta.columns.tolist())
display(meta.head())

In [ ]:
def resolve_final_image(row):
    path_columns = [
        "final_image_path", "image_path", "resolved_path",
        "synthetic_path", "generated_path", "generated_image_path",
        "output_path", "file_path"
    ]
    for col in path_columns:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
            raw = Path(str(row[col]).strip())
            candidates = [raw] if raw.is_absolute() else [PROJECT_ROOT / raw, FINAL_DIR / raw, raw]
            for candidate in candidates:
                if candidate.exists() and candidate.is_file():
                    return candidate.resolve()

    split_raw = str(row.get("split", "train")).strip().lower()
    split_map = {"training": "train", "train": "train", "validation": "val", "valid": "val", "val": "val", "testing": "test", "test": "test"}
    split = split_map.get(split_raw, split_raw)
    name = str(row.get("image_name", row.get("image_id", row.get("filename", "")))).strip()
    if not name or name.lower() == "nan":
        return None
    if Path(name).suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
        name += ".jpg"
    candidates = [
        FINAL_DIR / "images" / split / name,
        FINAL_DIR / "images" / name,
        FINAL_DIR / split / name,
        PROJECT_ROOT / "datasets" / "processed" / "images" / split / name,
    ]
    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate.resolve()
    # Last-resort exact filename search within the selected final dataset.
    if FINAL_DIR.exists():
        matches = list(FINAL_DIR.glob(f"**/{Path(name).name}"))
        if matches:
            return matches[0].resolve()
    return None

meta["resolved_path"] = meta.apply(resolve_final_image, axis=1)
meta = meta[meta["resolved_path"].notna()].copy()
meta["resolved_path"] = meta["resolved_path"].astype(str)

if "project_condition" in meta.columns:
    condition_col = "project_condition"
elif "condition" in meta.columns:
    condition_col = "condition"
    meta["project_condition"] = meta["condition"]
elif "weather" in meta.columns:
    condition_col = "weather"
    meta["project_condition"] = meta["weather"]
else:
    condition_col = None
assert condition_col is not None, "Condition column ('project_condition', 'condition', or 'weather') is required."

# Normalize common split and condition aliases while preserving original metadata.
meta["split_norm"] = meta["split"].astype(str).str.strip().str.lower().replace({
    "training": "train", "validation": "val", "valid": "val", "testing": "test"
}) if "split" in meta.columns else "train"
meta["condition_norm"] = meta[condition_col].astype(str).str.strip().str.lower().replace({
    "foggy": "fog", "mist": "fog", "misty": "fog"
})
print("Usable metadata rows with resolvable images:", len(meta))
print("Conditions:", meta["condition_norm"].value_counts(dropna=False).to_dict())
print("Splits:", meta["split_norm"].value_counts(dropna=False).to_dict())

## 2. Prepare real and synthetic Fog training data

The training source consists of:

1. Original real Fog images from `train`.
2. Accepted synthetic Fog images from the final training dataset, when available.

The validation source is **only original real Fog images from `val`**. This keeps the evaluation representative of real Fog rather than synthetic Fog.

In [ ]:
real_fog_train = meta[
    (meta["split_norm"] == "train") &
    (meta["condition_norm"] == "fog")
].copy()

real_fog_val = meta[
    (meta["split_norm"] == "val") &
    (meta["condition_norm"] == "fog")
].copy()

# Identify synthetic rows from common provenance columns used by Notebook 10.
synthetic_mask = pd.Series(
    False,
    index=meta.index
)

for col in [
    "is_synthetic",
    "synthetic",
    "source_type",
    "dataset_type",
    "origin",
    "source",
]:
    if col in meta.columns:
        values = meta[col].astype(str).str.lower()
        synthetic_mask |= (
            values.str.contains("synthetic", na=False) |
            values.str.contains("cyclegan", na=False) |
            values.isin(["true", "1", "yes"])
        )

synthetic_fog_train = meta[
    (meta["split_norm"] == "train") &
    (meta["condition_norm"] == "fog") &
    synthetic_mask
].copy()

# Fallback: if Notebook 10 metadata does not expose provenance,
# inspect the generated metadata produced by Notebook 08.
generated_meta_path = (
    PROJECT_ROOT /
    "datasets" /
    "processed" /
    "metadata" /
    "cyclegan_generated_metadata.csv"
)

if (
    USE_SYNTHETIC_FOG
    and len(synthetic_fog_train) == 0
    and generated_meta_path.exists()
):
    generated = pd.read_csv(generated_meta_path)

    path_candidates = [
        "generated_path",
        "image_path",
        "output_path",
        "generated_image_path",
    ]

    generated_path_col = next(
        (
            c for c in path_candidates
            if c in generated.columns
        ),
        None
    )

    if generated_path_col is not None:
        rows = []

        for _, r in generated.iterrows():
            p = Path(str(r[generated_path_col]))
            if not p.is_absolute():
                options = [PROJECT_ROOT / p, FINAL_DIR / p, p]
                p = next((q for q in options if q.exists()), p)

            if not p.exists() or not p.is_file():
                continue

            rows.append({
                "split": "train",
                "project_condition": "Fog",
                "resolved_path": str(p),
                "image_id": str(
                    r.get(
                        "image_id",
                        p.stem
                    )
                ),
                "image_name": p.name,
                "synthetic_source": "CycleGAN",
            })

        if rows:
            synthetic_fog_train = pd.DataFrame(rows)

if SYNTHETIC_LIMIT is not None:
    synthetic_fog_train = synthetic_fog_train.sample(
        min(
            SYNTHETIC_LIMIT,
            len(synthetic_fog_train)
        ),
        random_state=SEED
    )

train_sources = [
    real_fog_train[
        ["resolved_path"]
    ].assign(
        source_type="real_fog"
    )
]

if USE_SYNTHETIC_FOG and len(synthetic_fog_train):
    train_sources.append(
        synthetic_fog_train[
            ["resolved_path"]
        ].assign(
            source_type="synthetic_fog"
        )
    )

fog_train = pd.concat(
    train_sources,
    ignore_index=True
)

# Fail early rather than allowing an empty/shuffled DataLoader to fail later.
assert len(real_fog_train) > 0, "No real Fog training images found. Check metadata labels and train split."
assert len(real_fog_val) > 0, "No real Fog validation images found. Check metadata labels and val split."
assert len(fog_train) > 0, "No Fog training images available after combining sources."

if TRAIN_LIMIT is not None:
    fog_train = fog_train.sample(
        min(
            TRAIN_LIMIT,
            len(fog_train)
        ),
        random_state=SEED
    ).reset_index(drop=True)

if VAL_LIMIT is not None:
    real_fog_val = real_fog_val.sample(
        min(
            VAL_LIMIT,
            len(real_fog_val)
        ),
        random_state=SEED
    )

print("Real Fog train:", len(real_fog_train))
print("Synthetic Fog train:", len(synthetic_fog_train))
print("Combined Fog train:", len(fog_train))
print("Real Fog validation:", len(real_fog_val))

In [ ]:
print("Training source distribution:")

if "source_type" in fog_train.columns:
    display(
        fog_train["source_type"].value_counts()
    )

print("\nValidation condition:")
print(
    real_fog_val[
        condition_col
    ].value_counts()
    if condition_col in real_fog_val.columns
    else "Fog validation prepared."
)

## 3. Fog-to-clear pseudo-reference

Fog reduces contrast and visibility. The pseudo-reference therefore uses a conservative deterministic visibility-restoration operation:

- local smoothing to estimate low-frequency atmospheric veil,
- transmission-like contrast recovery,
- mild contrast and sharpness enhancement,
- clipping to a valid RGB range.

This is intentionally conservative because an aggressive synthetic target can create halos or artificial colors.

**Evaluation caveat:** these are algorithmically generated pseudo-targets, not paired clear-weather ground truth. PSNR/SSIM measure similarity to this heuristic target only; they do not prove real-world dehazing quality.

In [ ]:
def make_reference_target(img):
    img = img.convert("RGB")

    arr = np.asarray(img).astype(np.float32) / 255.0

    # Estimate low-frequency atmospheric veil.
    smooth_img = img.filter(
        ImageFilter.GaussianBlur(radius=4.0)
    )

    smooth = (
        np.asarray(smooth_img)
        .astype(np.float32) / 255.0
    )

    # Estimate scene contrast around the low-frequency veil.
    detail = arr - smooth

    # Conservative contrast recovery.
    restored = smooth + 1.55 * detail

    # Mild local/global contrast lift.
    restored = np.clip(
        restored,
        0.0,
        1.0
    )

    out = Image.fromarray(
        (restored * 255.0)
        .astype(np.uint8)
    )

    out = ImageEnhance.Contrast(
        out
    ).enhance(1.08)

    out = ImageEnhance.Sharpness(
        out
    ).enhance(1.12)

    return out


sample_path = fog_train.iloc[0]["resolved_path"]

sample_img = Image.open(
    sample_path
).convert("RGB")

sample_img = sample_img.resize(
    (IMAGE_SIZE, IMAGE_SIZE),
    Image.Resampling.BILINEAR
)

sample_target = make_reference_target(
    sample_img
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4)
)

axes[0].imshow(sample_img)
axes[0].set_title("Fog source")
axes[0].axis("off")

axes[1].imshow(sample_target)
axes[1].set_title("Pseudo-clear target")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class FogEnhancementDataset(Dataset):
    def __init__(self, frame, train=False):
        self.frame = frame.reset_index(
            drop=True
        )
        self.train = train

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        path = self.frame.iloc[idx][
            "resolved_path"
        ]

        img = Image.open(
            path
        ).convert("RGB")

        img = img.resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            Image.Resampling.BILINEAR
        )

        target = make_reference_target(
            img
        )

        if self.train and random.random() < 0.5:
            img = TF.hflip(img)
            target = TF.hflip(target)

        x = TF.to_tensor(img)
        y = TF.to_tensor(target)

        return (
            x,
            y,
            str(path)
        )


train_ds = FogEnhancementDataset(
    fog_train,
    train=True
)

val_ds = FogEnhancementDataset(
    real_fog_val,
    train=False
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

In [ ]:
# Sanity-check a batch before starting a potentially long training run.
assert len(train_ds) > 0, "Training dataset is empty."
assert len(val_ds) > 0, "Validation dataset is empty."
xb, yb, paths_b = next(iter(train_loader))
assert xb.ndim == 4 and xb.shape[1] == 3, f"Unexpected input batch shape: {tuple(xb.shape)}"
assert yb.shape == xb.shape, f"Input/target shape mismatch: {tuple(xb.shape)} vs {tuple(yb.shape)}"
assert torch.isfinite(xb).all() and torch.isfinite(yb).all(), "Non-finite values found in sample batch."
print("Batch sanity check passed:", tuple(xb.shape), "| sample:", paths_b[0])

## 4. Fog enhancement model

The architecture is kept consistent with the Night and Rain enhancement notebooks so all three condition-specific models have a compatible inference interface.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                3,
                padding=1
            ),
            nn.InstanceNorm2d(
                channels,
                affine=True
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                channels,
                channels,
                3,
                padding=1
            ),
            nn.InstanceNorm2d(
                channels,
                affine=True
            ),
        )

    def forward(self, x):
        return x + self.block(x)


class FogEnhancer(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(
                3,
                32,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        self.enc2 = nn.Sequential(
            nn.Conv2d(
                32,
                64,
                3,
                stride=2,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        self.enc3 = nn.Sequential(
            nn.Conv2d(
                64,
                128,
                3,
                stride=2,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        self.res = nn.Sequential(
            *[
                ResidualBlock(128)
                for _ in range(4)
            ]
        )

        self.dec2 = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        self.dec1 = nn.Sequential(
            nn.Conv2d(
                64,
                32,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        self.out = nn.Conv2d(
            32,
            3,
            3,
            padding=1
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        z = self.res(e3)

        z = F.interpolate(
            z,
            size=e2.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        z = self.dec2(
            z
        ) + e2

        z = F.interpolate(
            z,
            size=e1.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        z = self.dec1(
            z
        ) + e1

        residual = (
            torch.tanh(
                self.out(z)
            ) * 0.5
        )

        return torch.clamp(
            x + residual,
            0,
            1
        )


model = FogEnhancer().to(
    DEVICE
)

params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Trainable parameters: {params:,}"
)

## 5. Losses and metrics

The objective combines L1 reconstruction, SSIM-style structural similarity, and total variation. Validation is performed on real Fog images only.

In [ ]:
def gaussian_window(
    window_size=11,
    sigma=1.5,
    channels=3,
    device=None
):
    coords = (
        torch.arange(
            window_size,
            device=device
        ).float()
        - window_size // 2
    )

    g = torch.exp(
        -(coords ** 2) /
        (2 * sigma ** 2)
    )

    g = g / g.sum()

    kernel = torch.outer(
        g,
        g
    )

    return kernel.expand(
        channels,
        1,
        window_size,
        window_size
    ).contiguous()


def ssim_torch(
    x,
    y,
    window_size=11
):
    channels = x.shape[1]

    window = gaussian_window(
        window_size,
        1.5,
        channels,
        x.device
    )

    mu_x = F.conv2d(
        x,
        window,
        padding=window_size // 2,
        groups=channels
    )

    mu_y = F.conv2d(
        y,
        window,
        padding=window_size // 2,
        groups=channels
    )

    mu_x2 = mu_x * mu_x
    mu_y2 = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x = (
        F.conv2d(
            x * x,
            window,
            padding=window_size // 2,
            groups=channels
        )
        - mu_x2
    )

    sigma_y = (
        F.conv2d(
            y * y,
            window,
            padding=window_size // 2,
            groups=channels
        )
        - mu_y2
    )

    sigma_xy = (
        F.conv2d(
            x * y,
            window,
            padding=window_size // 2,
            groups=channels
        )
        - mu_xy
    )

    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    score = (
        (2 * mu_xy + C1)
        * (2 * sigma_xy + C2)
        /
        (
            (mu_x2 + mu_y2 + C1)
            * (sigma_x + sigma_y + C2)
            + 1e-8
        )
    )

    return score.mean()


def total_variation(x):
    return (
        (
            x[:, :, :, 1:]
            - x[:, :, :, :-1]
        ).abs().mean()
        +
        (
            x[:, :, 1:, :]
            - x[:, :, :-1, :]
        ).abs().mean()
    )


def psnr_torch(x, y):
    mse = F.mse_loss(
        x,
        y
    ).item()

    return 10 * math.log10(
        1.0 /
        max(mse, 1e-12)
    )


def compute_loss(
    pred,
    target
):
    l1 = F.l1_loss(
        pred,
        target
    )

    structural = (
        1.0
        - ssim_torch(
            pred,
            target
        )
    )

    tv = total_variation(
        pred
    )

    return (
        l1
        + 0.20 * structural
        + 0.01 * tv
    )

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

try:
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=torch.cuda.is_available()
    )
except Exception:
    scaler = torch.cuda.amp.GradScaler(
        enabled=torch.cuda.is_available()
    )


def amp_context():
    if torch.cuda.is_available():
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        )

    return torch.autocast(
        device_type="cpu",
        dtype=torch.bfloat16,
        enabled=False
    )

## 6. Training

The best checkpoint is selected using validation loss on **real Fog validation images**. Intermediate checkpoints are saved every `SAVE_EVERY` epochs.

In [ ]:
FOG_MODEL_DIR = MODEL_DIR / "fog"

FOG_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

history = []
best_val = float("inf")
bad_epochs = 0

best_path = (
    FOG_MODEL_DIR /
    "fog_enhancer_best.pth"
)

last_path = (
    FOG_MODEL_DIR /
    "fog_enhancer_last.pth"
)

for epoch in range(
    1,
    EPOCHS + 1
):
    start = time.time()

    model.train()
    train_losses = []

    for x, y, _ in train_loader:
        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with amp_context():
            pred = model(x)
            loss = compute_loss(
                pred,
                y
            )

        if torch.cuda.is_available():
            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        train_losses.append(
            loss.item()
        )

    model.eval()

    val_losses = []
    val_psnr = []
    val_ssim = []

    with torch.no_grad():
        for x, y, _ in val_loader:
            x = x.to(
                DEVICE,
                non_blocking=True
            )

            y = y.to(
                DEVICE,
                non_blocking=True
            )

            with amp_context():
                pred = model(x)
                loss = compute_loss(
                    pred,
                    y
                )

            pred_f = pred.float()
            y_f = y.float()

            val_losses.append(
                loss.item()
            )

            val_psnr.append(
                psnr_torch(
                    pred_f,
                    y_f
                )
            )

            val_ssim.append(
                ssim_torch(
                    pred_f,
                    y_f
                ).item()
            )

    train_loss = float(
        np.mean(
            train_losses
        )
    )

    val_loss = float(
        np.mean(
            val_losses
        )
    )

    mean_psnr = float(
        np.mean(
            val_psnr
        )
    )

    mean_ssim = float(
        np.mean(
            val_ssim
        )
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_psnr": mean_psnr,
        "val_ssim": mean_ssim,
        "lr": optimizer.param_groups[0]["lr"],
        "seconds": time.time() - start,
    }

    history.append(
        row
    )

    scheduler.step(
        val_loss
    )

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_psnr": mean_psnr,
        "val_ssim": mean_ssim,
        "config": {
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "seed": SEED,
            "use_synthetic_fog": USE_SYNTHETIC_FOG,
        },
    }

    torch.save(
        checkpoint,
        last_path
    )

    if epoch % SAVE_EVERY == 0:
        torch.save(
            model.state_dict(),
            FOG_MODEL_DIR /
            f"fog_enhancer_epoch_{epoch:03d}.pth"
        )

    if val_loss < best_val:
        best_val = val_loss
        bad_epochs = 0

        torch.save(
            checkpoint,
            best_path
        )
    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train {train_loss:.4f} | "
        f"val {val_loss:.4f} | "
        f"PSNR {mean_psnr:.2f} | "
        f"SSIM {mean_ssim:.4f}"
    )

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

In [ ]:
history_df = pd.DataFrame(
    history
)

history_path = (
    RESULTS_DIR /
    "fog_enhancement_training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

config = {
    "project_root": str(PROJECT_ROOT),
    "task": "fog_enhancement_training",
    "source_condition": "Fog",
    "real_fog_train_count": int(
        len(real_fog_train)
    ),
    "synthetic_fog_train_count": int(
        len(synthetic_fog_train)
    ),
    "combined_train_count": int(
        len(fog_train)
    ),
    "real_fog_validation_count": int(
        len(real_fog_val)
    ),
    "test_used_for_training": False,
    "synthetic_fog_enabled": USE_SYNTHETIC_FOG,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_requested": EPOCHS,
    "epochs_completed": int(
        len(history_df)
    ),
    "learning_rate": LEARNING_RATE,
    "device": str(DEVICE),
    "best_checkpoint": str(best_path),
    "last_checkpoint": str(last_path),
    "target_type": "deterministic_pseudo_reference",
}

with open(
    RESULTS_DIR /
    "fog_enhancement_training_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config,
        f,
        indent=2
    )

print(
    "Best checkpoint:",
    best_path
)

print(
    "Last checkpoint:",
    last_path
)

print(
    "History:",
    history_path
)

## 7. Training curves

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train loss"
)

ax.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Val loss"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Fog enhancement training")
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    history_df["epoch"],
    history_df["val_psnr"],
    label="Val PSNR"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("PSNR (dB)")
ax.set_title("Real Fog validation PSNR")
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 8. Visual sanity check

Load the best checkpoint and compare the Fog input, model output, and pseudo-clear reference on real validation Fog images.

In [ ]:
checkpoint = torch.load(
    best_path,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

n_show = min(
    4,
    len(val_ds)
)

fig, axes = plt.subplots(
    n_show,
    3,
    figsize=(12, 4 * n_show)
)

if n_show == 1:
    axes = np.expand_dims(
        axes,
        axis=0
    )

for i in range(n_show):
    x, y, path = val_ds[i]

    with torch.no_grad():
        pred = model(
            x.unsqueeze(0).to(DEVICE)
        ).squeeze(0).cpu().clamp(
            0,
            1
        )

    axes[i, 0].imshow(
        x.permute(1, 2, 0)
    )
    axes[i, 0].set_title(
        "Fog input"
    )
    axes[i, 0].axis("off")

    axes[i, 1].imshow(
        pred.permute(1, 2, 0)
    )
    axes[i, 1].set_title(
        "Enhanced output"
    )
    axes[i, 1].axis("off")

    axes[i, 2].imshow(
        y.permute(1, 2, 0)
    )
    axes[i, 2].set_title(
        "Pseudo-clear reference"
    )
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

## 9. Final summary

Main artifact:

`models/enhancement/fog/fog_enhancer_best.pth`

Important: the validation set contains only real Fog images, while synthetic Fog is used only for training augmentation. Use downstream detection/tracking evaluation to determine whether Fog enhancement improves the complete Adaptive_Road_Vision system.

In [ ]:
if len(history_df):
    best_idx = history_df[
        "val_loss"
    ].idxmin()

    summary = {
        "best_epoch": int(
            history_df.loc[
                best_idx,
                "epoch"
            ]
        ),
        "best_val_loss": float(
            history_df.loc[
                best_idx,
                "val_loss"
            ]
        ),
        "best_val_psnr": float(
            history_df.loc[
                best_idx,
                "val_psnr"
            ]
        ),
        "best_val_ssim": float(
            history_df.loc[
                best_idx,
                "val_ssim"
            ]
        ),
        "checkpoint": str(
            best_path
        ),
        "real_fog_train_images": int(
            len(real_fog_train)
        ),
        "synthetic_fog_train_images": int(
            len(synthetic_fog_train)
        ),
        "validation_images": int(
            len(real_fog_val)
        ),
    }
else:
    summary = {
        "error":
            "No training history was produced."
    }

with open(
    RESULTS_DIR /
    "fog_enhancement_training_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(
    json.dumps(
        summary,
        indent=2
    )
)